# Auriga Capital — Quantitative Researcher Assessment
### Options & Signal Research

---

**Instructions**

This assessment is divided into two parts:
- **Part A: Options Pricing & Volatility Surface** 
- **Part B: Signal Research & Alpha Generation** 

You may use any standard Python libraries (`numpy`, `scipy`, `pandas`, `matplotlib`, `sklearn`, etc.). Do **not** use dedicated options pricing libraries (e.g. `QuantLib`, `mibian`, `py_vollib`) — we want to see your implementations.

Please leave your code well-commented and show your reasoning in markdown cells where prompted. We value clarity of thought as much as correctness.

Submit as a completed `.ipynb` with all cells run and output visible.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.stats import norm
from scipy.optimize import brentq, minimize
from scipy.interpolate import CubicSpline
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

---
## Part A: Options Pricing & Volatility Surface

---

### A1. Black-Scholes Foundations

Implement the core Black-Scholes machinery from scratch. You will build on this throughout Part A.

In [ ]:
# A1a. Implement the Black-Scholes pricing formula for European calls and puts.
# Parameters:
#   S     - spot price
#   K     - strike
#   T     - time to expiry in years
#   r     - risk-free rate (continuously compounded)
#   sigma - annualised volatility
#   q     - continuous dividend / convenience yield (default 0)
#   flag  - 'call' or 'put'

def bs_price(S, K, T, r, sigma, q=0.0, flag='call'):
    """
    Black-Scholes price for a European option.
    Returns NaN for degenerate inputs (T<=0, sigma<=0).
    """
    # YOUR CODE HERE
    pass


# A1b. Implement a vectorised implied volatility solver.
# Use Brent's method (scipy.optimize.brentq) as the root-finder.
# Handle edge cases: intrinsic-only prices, negative time value, etc.

def implied_vol(price, S, K, T, r, q=0.0, flag='call',
                vol_lo=1e-6, vol_hi=5.0, tol=1e-8):
    """
    Implied volatility via Brent's method.
    Returns np.nan if no solution exists within [vol_lo, vol_hi].
    """
    # YOUR CODE HERE
    pass


# A1c. Implement ALL six first- and second-order Greeks:
#   delta, gamma, vega, theta, rho, vanna (dDelta/dVol)
# Return as a dict.

def bs_greeks(S, K, T, r, sigma, q=0.0, flag='call'):
    """
    Returns dict with keys: delta, gamma, vega, theta, rho, vanna.
    Theta should be in daily P&L terms (i.e. divided by 365).
    """
    # YOUR CODE HERE
    pass


# ---- Sanity checks (do not modify) ----
# Put-call parity
S, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.20
c = bs_price(S, K, T, r, sigma, flag='call')
p = bs_price(S, K, T, r, sigma, flag='put')
parity_error = abs((c - p) - (S - K * np.exp(-r * T)))
print(f"Put-call parity error: {parity_error:.2e}  (should be < 1e-10)")

# IV round-trip
iv = implied_vol(c, S, K, T, r, flag='call')
print(f"IV round-trip error:   {abs(iv - sigma):.2e}  (should be < 1e-6)")

---
### A2. Volatility Surface Calibration

Below is a snapshot of SPX option market data. Your task is to build a no-arbitrage implied volatility surface.

The data represents mid implied vols (annualised) across strikes and expiries.

In [ ]:
# Market data — do not modify
SPOT = 4500.0
RATE = 0.053

# Expiries in years
EXPIRIES = np.array([1/52, 2/52, 1/12, 2/12, 3/12, 6/12, 1.0, 1.5, 2.0])

# Strikes expressed as % of spot (moneyness)
MONEYNESS = np.array([0.70, 0.75, 0.80, 0.85, 0.90, 0.925, 0.95,
                       0.975, 1.00, 1.025, 1.05, 1.075, 1.10, 1.15, 1.20])

# Implied vol grid [expiry x moneyness] — mid-market, annualised
# fmt: off
IV_GRID = np.array([
 # .70   .75   .80   .85   .90  .925   .95  .975  1.00  1.025  1.05  1.075  1.10  1.15  1.20
 [0.55, 0.49, 0.43, 0.37, 0.31, 0.28, 0.25, 0.22, 0.20,  0.21, 0.23,  0.26, 0.29, 0.35, 0.42],  # 1w
 [0.52, 0.46, 0.40, 0.35, 0.29, 0.27, 0.24, 0.21, 0.19,  0.20, 0.22,  0.25, 0.27, 0.33, 0.40],  # 2w
 [0.48, 0.43, 0.38, 0.33, 0.28, 0.26, 0.23, 0.20, 0.18,  0.19, 0.21,  0.23, 0.26, 0.31, 0.38],  # 1m
 [0.45, 0.40, 0.36, 0.31, 0.27, 0.25, 0.22, 0.20, 0.18,  0.19, 0.20,  0.22, 0.25, 0.30, 0.36],  # 2m
 [0.43, 0.38, 0.34, 0.30, 0.26, 0.24, 0.22, 0.19, 0.18,  0.19, 0.20,  0.22, 0.24, 0.29, 0.35],  # 3m
 [0.40, 0.36, 0.32, 0.28, 0.25, 0.23, 0.21, 0.19, 0.18,  0.19, 0.20,  0.21, 0.23, 0.27, 0.33],  # 6m
 [0.37, 0.34, 0.30, 0.27, 0.24, 0.22, 0.20, 0.19, 0.18,  0.18, 0.19,  0.21, 0.22, 0.26, 0.31],  # 1y
 [0.36, 0.33, 0.29, 0.26, 0.23, 0.21, 0.20, 0.18, 0.17,  0.18, 0.19,  0.20, 0.22, 0.25, 0.30],  # 1.5y
 [0.35, 0.32, 0.28, 0.25, 0.22, 0.21, 0.19, 0.18, 0.17,  0.17, 0.18,  0.20, 0.21, 0.24, 0.29],  # 2y
])
# fmt: on

STRIKES = SPOT * MONEYNESS
print(f"Surface shape: {IV_GRID.shape}  ({len(EXPIRIES)} expiries x {len(MONEYNESS)} strikes)")

In [ ]:
# A2a. Visualise the raw implied vol surface.
# Plot IV as a function of moneyness for each expiry on a single chart.
# Comment on the key features you observe (smile/skew shape, term structure).

# YOUR CODE HERE

In [ ]:
# A2b. Arbitrage detection.
#
# Two fundamental no-arbitrage conditions must hold on any valid surface:
#
#  (i)  Calendar spread arbitrage: for a fixed strike, option prices must be
#       non-decreasing in expiry (total variance must be monotonically increasing).
#
#  (ii) Butterfly arbitrage: for a fixed expiry, the local vol implied by the
#       surface must be non-negative everywhere. This is equivalent to checking
#       that d²C/dK² >= 0 (the risk-neutral density is non-negative).
#
# Tasks:
#   1. Write a function that checks BOTH conditions on the IV_GRID data.
#   2. Report any violations found (expiry, moneyness, severity).
#   3. In the markdown cell below, explain what each violation means economically.

def check_calendar_arbitrage(iv_grid, expiries, moneyness):
    """
    Returns list of (expiry_idx_i, expiry_idx_j, moneyness_idx, violation_magnitude)
    for all detected calendar spread violations.
    """
    # YOUR CODE HERE
    pass


def check_butterfly_arbitrage(iv_grid, expiries, moneyness, spot, rate):
    """
    Returns list of (expiry_idx, moneyness_idx, negative_density_value)
    for all detected butterfly violations.
    Hint: compute d²C/dK² numerically for each expiry slice.
    """
    # YOUR CODE HERE
    pass


cal_violations  = check_calendar_arbitrage(IV_GRID, EXPIRIES, MONEYNESS)
bfly_violations = check_butterfly_arbitrage(IV_GRID, EXPIRIES, MONEYNESS, SPOT, RATE)

print(f"Calendar violations:  {len(cal_violations)}")
print(f"Butterfly violations: {len(bfly_violations)}")

**A2b — Economic interpretation of violations:**

*Your answer here.*

In [ ]:
# A2c. SVI (Stochastic Volatility Inspired) slice fit.
#
# The raw SVI parameterisation for total implied variance w(k) as a function of
# log-moneyness k = log(K/F) is:
#
#   w(k) = a + b * [ rho*(k - m) + sqrt((k - m)^2 + sigma^2) ]
#
# where {a, b, rho, m, sigma} are the five SVI parameters.
#
# Constraints for no-butterfly-arbitrage on a single slice:
#   b >= 0,  |rho| < 1,  sigma > 0,  a + b*sigma*sqrt(1-rho^2) >= 0
#
# Tasks:
#   1. Implement the SVI total variance formula.
#   2. Fit SVI to EACH expiry slice by minimising weighted least-squares
#      against the market implied vols. Use sensible initial guesses.
#   3. Plot fitted vs market IV for the 3m, 6m, and 1y slices.
#   4. Report the per-slice RMSE in vol points (bps).

def svi_total_variance(k, a, b, rho, m, sigma):
    """Raw SVI total variance w(k)."""
    # YOUR CODE HERE
    pass


def fit_svi_slice(iv_slice, expiry, moneyness, spot, rate):
    """
    Fit SVI to one expiry slice.
    Returns (params_dict, rmse_in_vol_bps).
    """
    # YOUR CODE HERE
    pass


# Fit all slices and store results
svi_results = []
for i, T in enumerate(EXPIRIES):
    params, rmse = fit_svi_slice(IV_GRID[i], T, MONEYNESS, SPOT, RATE)
    svi_results.append({'T': T, 'params': params, 'rmse_bps': rmse})
    print(f"T={T:.4f}y  RMSE={rmse:.1f} bps")

In [ ]:
# A2d. Local volatility surface via Dupire's formula.
#
# Given a smooth implied vol surface sigma(K, T), Dupire's equation gives
# the local volatility sigma_loc(S, T) consistent with all option prices:
#
#   sigma_loc^2(K,T) = [ dC/dT ] / [ 0.5 * K^2 * d^2C/dK^2 ]
#
# Tasks:
#   1. Using your SVI fits, construct a smooth continuous surface over a
#      fine (K, T) grid. Interpolate SVI parameters across expiries.
#   2. Compute the Dupire local vol surface numerically.
#   3. Plot the local vol surface as a heatmap.
#   4. Comment on how local vol compares to implied vol, and what this
#      implies for the dynamics of the underlying.

# Fine grid for surface evaluation
T_FINE = np.linspace(EXPIRIES[0], EXPIRIES[-1], 80)
K_FINE = np.linspace(SPOT * 0.72, SPOT * 1.18, 80)

# YOUR CODE HERE

**A2d — Local vol vs implied vol dynamics:**

*Your answer here.*

---
### A3. Option-Implied Information & Signal Construction

This section bridges Part A and Part B. You will extract option-implied quantities and evaluate their predictive content for underlying returns.

In [ ]:
# Synthetic daily time-series data — do not modify
np.random.seed(42)
N = 756  # ~3 years of daily data

dates = pd.date_range('2021-01-04', periods=N, freq='B')

# Simulate a mean-reverting vol process (Heston-like)
kappa, theta_v, xi = 3.0, 0.04, 0.35
v = np.zeros(N); v[0] = 0.04
for t in range(1, N):
    v[t] = max(v[t-1] + kappa*(theta_v - v[t-1])/252
               + xi*np.sqrt(v[t-1]/252)*np.random.randn(), 1e-6)

# Spot price with leverage effect (negative correlation between spot and vol)
rho_sv = -0.65
spot_returns = np.zeros(N)
for t in range(1, N):
    dW_v = np.random.randn()
    dW_s = rho_sv*dW_v + np.sqrt(1-rho_sv**2)*np.random.randn()
    spot_returns[t] = -0.0002 + np.sqrt(v[t-1]/252)*dW_s

spot = pd.Series(4000 * np.exp(np.cumsum(spot_returns)), index=dates, name='spot')

# Implied vol proxies (ATM IV for different tenors)
noise = lambda s=0.01: s * np.random.randn(N)
iv_1m  = pd.Series(np.sqrt(v) + 0.02 + noise(0.005), index=dates, name='iv_1m')
iv_3m  = pd.Series(np.sqrt(v) + 0.025 + noise(0.004), index=dates, name='iv_3m')
iv_6m  = pd.Series(np.sqrt(v) + 0.03 + noise(0.003), index=dates, name='iv_6m')
iv_12m = pd.Series(np.sqrt(v) + 0.035 + noise(0.003), index=dates, name='iv_12m')

# 25-delta risk reversal (skew proxy) and butterfly (kurtosis proxy)
risk_reversal = pd.Series(-0.08*np.sqrt(v) - 0.01 + noise(0.004), index=dates, name='rr_25d')
butterfly     = pd.Series( 0.03*np.sqrt(v) + 0.005 + noise(0.002), index=dates, name='bfly_25d')

# Realised vol (20-day rolling)
rv_20d = spot.pct_change().rolling(20).std() * np.sqrt(252)
rv_20d.name = 'rv_20d'

df = pd.concat([spot, iv_1m, iv_3m, iv_6m, iv_12m,
                risk_reversal, butterfly, rv_20d], axis=1).dropna()
print(df.shape, df.head())

In [ ]:
# A3a. Variance risk premium (VRP).
#
# The VRP is the compensation investors earn for bearing variance risk.
# A common definition: VRP(t) = IV(t)^2 - E[RV(t, t+h)]
#
# Tasks:
#   1. Compute the 1-month VRP using iv_1m and forward-looking 21-day realised vol.
#   2. Plot VRP over time. Annotate any notable regimes.
#   3. Report the mean, median, and proportion of days where VRP > 0.
#   4. In the markdown cell below, explain WHY the VRP is typically positive
#      and what drives its time-variation.

# YOUR CODE HERE

**A3a — Economic interpretation of the VRP:**

*Your answer here.*

In [ ]:
# A3b. Term structure carry signal.
#
# The vol term structure slope often contains information about
# forward-looking vol dynamics and risk premia.
#
# Tasks:
#   1. Construct a term structure slope measure:
#      slope(t) = iv_12m(t) - iv_1m(t)
#   2. Compute the 1-month forward change in ATM IV:
#      delta_iv(t) = iv_1m(t+21) - iv_1m(t)
#   3. Run a simple OLS regression of delta_iv on slope.
#      Report the coefficient, t-stat, and R².
#   4. Design a simple long/short signal based on slope.
#      Backtest this signal: go long/short a straddle at the 1m expiry,
#      hold for 21 days, and compute PnL assuming you are delta-hedged daily
#      (i.e. your P&L is driven by vega * delta_iv).
#   5. Report annualised Sharpe ratio, hit rate, and plot the cumulative PnL.

# YOUR CODE HERE

In [ ]:
# A3c. Skew signal.
#
# The 25-delta risk reversal (rr_25d) captures the implied skewness of
# the return distribution. An unusually negative RR may indicate
# hedging demand in puts, or forecast negative future spot returns.
#
# Tasks:
#   1. Z-score the risk_reversal series using a 60-day rolling mean and std.
#   2. Compute 5-day and 21-day forward spot returns.
#   3. Test whether the z-scored RR predicts spot returns at both horizons:
#      - Report IC (information coefficient = rank correlation), t-stat.
#      - Plot rolling 60-day IC over time.
#   4. Combine the VRP signal (A3a) and the skew z-score into a composite signal.
#      Weight them by their trailing 60-day IC. Backtest and compare
#      Sharpe to each individual signal.
#
# Note: keep backtesting simple — daily rebalancing, equal-dollar notional,
# no transaction costs required (but comment on where they would matter).

# YOUR CODE HERE

---
## Part B: Signal Research & Alpha Generation

---

### B1. Roll Dynamics & Futures Carry

Systematic carry strategies exploit the premium embedded in futures term structures. This section tests your ability to model, extract, and evaluate carry signals across instruments.

In [ ]:
# Synthetic futures term structure data — do not modify
np.random.seed(7)
M = 504  # ~2 years daily
fdates = pd.date_range('2022-01-03', periods=M, freq='B')

# Two instruments: an equity index future (EQ) and a commodity future (CM)
# Maturities: front (F1), 2nd (F2), 3rd (F3), 4th (F4)

# EQ: mild backwardation on average, strong mean-reversion in basis
eq_spot = 4000 * np.exp(np.cumsum(0.0003 + 0.012/np.sqrt(252) * np.random.randn(M)))
eq_carry_mean = -0.002  # slightly negative (index futures typically trade at discount)
eq_carry = eq_carry_mean + 0.003 * np.random.randn(M)
eq_f1 = eq_spot * np.exp(eq_carry * 1/12)
eq_f2 = eq_spot * np.exp(eq_carry * 2/12 + 0.001*np.random.randn(M))
eq_f3 = eq_spot * np.exp(eq_carry * 3/12 + 0.002*np.random.randn(M))
eq_f4 = eq_spot * np.exp(eq_carry * 4/12 + 0.003*np.random.randn(M))

# CM: persistent contango, with occasional backwardation spikes
cm_spot = 80 * np.exp(np.cumsum(0.0001 + 0.018/np.sqrt(252) * np.random.randn(M)))
cm_carry_base = 0.004 + 0.0015 * np.random.randn(M)
shock_days = np.random.choice(M, 15, replace=False)
cm_carry_base[shock_days] -= 0.015  # backwardation shock events
cm_carry = np.clip(cm_carry_base, -0.02, 0.015)
cm_f1 = cm_spot * np.exp(cm_carry * 1/12)
cm_f2 = cm_spot * np.exp(cm_carry * 2/12 + 0.002*np.random.randn(M))
cm_f3 = cm_spot * np.exp(cm_carry * 3/12 + 0.003*np.random.randn(M))
cm_f4 = cm_spot * np.exp(cm_carry * 4/12 + 0.004*np.random.randn(M))

futures = pd.DataFrame({
    'eq_f1': eq_f1, 'eq_f2': eq_f2, 'eq_f3': eq_f3, 'eq_f4': eq_f4,
    'cm_f1': cm_f1, 'cm_f2': cm_f2, 'cm_f3': cm_f3, 'cm_f4': cm_f4,
}, index=fdates)

print(futures.head())

In [ ]:
# B1a. Compute and visualise roll yield.
#
# Roll yield is the return earned by holding a futures contract as it converges
# towards spot, assuming the term structure shape is unchanged.
#
# Tasks:
#   1. Define a clean measure of roll yield for each instrument.
#      Use annualised terms so they are comparable across tenors.
#   2. Plot the roll yield time series for both EQ and CM.
#   3. For CM, identify and annotate the backwardation shock events.
#   4. In the markdown cell, explain the difference between roll yield,
#      spot return, and total futures return.

# YOUR CODE HERE

**B1a — Roll yield, spot return, and total futures return:**

*Your answer here.*

In [ ]:
# B1b. Carry signal and backtest.
#
# Tasks:
#   1. Construct a daily carry signal for each instrument:
#      - Positive signal = curve in backwardation (short front, long back)
#      - Negative signal = curve in contango (long front, short back)
#      Normalise so signals have unit volatility over a trailing 60-day window.
#
#   2. Backtest the carry trade:
#      - Position in F1 proportional to signal, hedge with F2 for a
#        duration-neutral calendar spread.
#      - Use next-day open prices (i.e. signal at t trades at t+1 prices).
#      - Assume a round-trip cost of 0.5bp per leg.
#
#   3. Report per-instrument and combined (equal-weighted) performance:
#      Sharpe, Sortino, max drawdown, hit rate, turnover (daily).
#
#   4. Plot the underwater curve (drawdown over time) for the combined strategy.

# YOUR CODE HERE

---
### B2. Signal Evaluation & Research Rigour

A critical skill in systematic research is distinguishing genuine alpha from noise. This section tests statistical discipline.

In [ ]:
# B2a. Multiple testing and the deflated Sharpe ratio.
#
# Suppose a researcher has tested 50 strategies over a 2-year sample (504 days).
# The best strategy has an observed Sharpe ratio of 1.8.
#
# Tasks:
#   1. Implement the Deflated Sharpe Ratio (DSR) framework from Bailey & Lopez de Prado.
#      The key formula is:
#
#        SR* = SR_benchmark  (the maximum expected Sharpe under H0 with N trials)
#        DSR = Phi[ (SR_obs - SR*) * sqrt(T-1) / sqrt(1 - gamma3*SR_obs + (gamma4-1)/4 * SR_obs^2) ]
#
#      where gamma3 is skewness and gamma4 is excess kurtosis of returns,
#      and SR* ≈ (1 - euler_gamma) * Z(1 - 1/N) + euler_gamma * Z(1 - 1/(N*e))
#      (Z = normal quantile, euler_gamma ≈ 0.5772)
#
#   2. Compute the DSR for the scenario above. Assume returns are normally distributed
#      (gamma3=0, gamma4=0).
#   3. What is the minimum observed Sharpe such that DSR > 0.95?
#   4. In the markdown cell, discuss the practical implications for how you would
#      structure a research process to control for multiple testing.

def deflated_sharpe_ratio(sr_obs, n_trials, n_obs, skew=0.0, kurt=0.0):
    """
    Compute the Deflated Sharpe Ratio.
    sr_obs  : observed annualised Sharpe ratio
    n_trials: number of strategies tested
    n_obs   : number of daily observations in the sample
    skew    : skewness of daily returns
    kurt    : excess kurtosis of daily returns
    Returns the DSR (a probability between 0 and 1).
    """
    # YOUR CODE HERE
    pass


dsr = deflated_sharpe_ratio(sr_obs=1.8, n_trials=50, n_obs=504)
print(f"DSR: {dsr:.4f}")

**B2a — Research process implications:**

*Your answer here.*

In [ ]:
# B2b. Regime detection and signal conditioning.
#
# Many signals have regime-dependent performance. A good researcher identifies
# WHEN a signal works and why — not just whether it works on average.
#
# Using the data from Part A (df dataframe with spot, IV, RV, etc.):
#
# Tasks:
#   1. Define at least two distinct market regimes (e.g. by vol level, vol-of-vol,
#      or term structure slope). Justify your choice.
#
#   2. Re-evaluate the composite signal from A3c conditional on each regime.
#      Report IC, hit rate, and Sharpe per regime.
#
#   3. Build a simple regime-aware version of the signal:
#      scale signal size by a regime-conditional confidence weight.
#
#   4. Does conditioning improve risk-adjusted returns? Report and discuss.

# YOUR CODE HERE

---
### B3. Open-Ended Research Question

This is the most important section. There are no right or wrong answers — we are simply evaluating the quality of your thinking.

**B3. Design a novel signal.**

Propose and implement (or sketch in detail if implementation is infeasible in this notebook) a signal that uses option-implied information to predict **either** (a) the direction of the underlying, or (b) future realised volatility, or (c) volatility risk premia — at a 5–21 day horizon.

Your response should cover:

1. **Economic hypothesis**: What market inefficiency or structural behaviour does this signal exploit? Why should it persist?
2. **Signal construction**: How exactly would you compute it from observable data? What normalisation or transformations would you apply?
3. **Evaluation framework**: How would you test it rigorously? What data would you need? What are the risks of overfitting?
4. **Risk considerations**: What market conditions would make this signal fail? How would you size and hedge a position based on it?
5. **Implementation**: Provide working code or, if data is unavailable, a detailed pseudocode/outline.

*Your answer here — use as many code and markdown cells as you need.*

---
## Submission Checklist

Before submitting, please confirm:

- [ ] All cells run top-to-bottom without errors
- [ ] All markdown cells are completed with your reasoning
- [ ] Plots are visible with labels and titles
- [ ] Code is commented and readable
- [ ] B3 open-ended question answered in detail

We look forward to reviewing your work.

---
*Auriga Capital — Confidential*